# Cargar documentos a Redis (RAG)

Notebook de **un solo proposito**: subir uno o mas documentos (PDF, TXT, MD, DOCX, CSV) al Redis del curso, listos para usarse como corpus RAG.

Esta version carga el corpus del asistente de criptomonedas con **dos documentos**:

- `criptomonedas-desde-cero.md`
- `Criptoactivos_Basico.pdf`

El corpus queda preparado para **busqueda vectorial KNN en Redis/RediSearch**. Los fragmentos se guardan como hashes y el embedding se almacena en formato binario `FLOAT32`, dentro de un indice HNSW con metrica coseno.

**Como se usa:**
1. Ejecuta las celdas 1 a 3 (instalacion, conexion, funciones).
2. Revisa la celda **Parametros**: ahi estan los documentos, la base `ebook_cripto` y el chunking.
3. Ejecuta la celda de carga. Al final hay una prueba de recuperacion KNN hecha por Redis.

**Evidencia para la pauta:** el notebook imprime corpus, base, indice exacto, embeddings, campos, dimensiones, metrica, algoritmo, cantidad de fragmentos y fuentes.


## 1. Instalación de dependencias

In [ ]:
# Descomenta si faltan librerías:
# %pip install -q redis langchain langchain-community langchain-google-genai pypdf python-dotenv docx2txt

import sys
print("Python:", sys.version)

## 2. Conexión a Redis y Gemini

Las credenciales se leen del `.env` (en esta misma carpeta). Variables:
`REDIS_URL`, `REDIS_PREFIJO`, `GOOGLE_API_KEY`.

In [ ]:
import os, re, json, hashlib
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import redis
from redis.commands.search.field import TextField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.query import Query
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()  # busca el .env en esta carpeta

REDIS_URL = os.environ["REDIS_URL"]
PREFIJO   = os.environ["REDIS_PREFIJO"]

R = redis.Redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=8)
print("PING ->", R.ping(), "| prefijo:", PREFIJO)

# Mismo modelo de embeddings usado por el asistente.
MODELO_EMBEDDINGS = "models/gemini-embedding-001"
DIM_EMBEDDINGS = 3072
CAMPO_VECTOR = "vector"
DISTANCIA = "COSINE"
ALGORITMO_VECTORIAL = "HNSW"

emb = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDINGS)
print("Conexion lista (Redis + Gemini + RediSearch).")
print("Embeddings:", MODELO_EMBEDDINGS, "| dim:", DIM_EMBEDDINGS, "| indice:", ALGORITMO_VECTORIAL, DISTANCIA)


## 3. Funciones: leer, chunkear, guardar e indexar

- `cargar_documento(ruta)` detecta el formato por extension y devuelve paginas o secciones como `Document` con su fuente.
- `dividir_texto(...)` parte el texto en fragmentos.
- `guardar_fragmentos(base, fragmentos)` sube cada fragmento a Redis y crea un indice RediSearch KNN.
- El campo vectorial se guarda como bytes `FLOAT32`, que es el formato que RediSearch necesita para `KNN`.


In [ ]:
def dividir_texto(texto, size=500, overlap=130, seps=("\n\n", ". ", " ", "")):
    """Divide texto en chunks con solapamiento (equivalente a un text splitter)."""
    if len(texto) <= size:
        return [texto] if texto.strip() else []
    sep = next((s for s in seps if s and s in texto), "")
    partes = texto.split(sep) if sep else list(texto)
    chunks, actual = [], ""
    for parte in partes:
        cand = parte if not actual else actual + sep + parte
        if len(cand) <= size:
            actual = cand
        else:
            if actual:
                chunks.append(actual)
            actual = (actual[-overlap:] + sep + parte) if actual else parte
            while len(actual) > size:
                chunks.append(actual[:size])
                actual = actual[size - overlap:]
    if actual.strip():
        chunks.append(actual)
    return chunks


def cargar_documento(ruta):
    """Lee un documento de cualquier formato soportado y devuelve una lista de
    Document con su 'fuente'. Formatos: .pdf .txt .md .csv .docx"""
    ruta = str(ruta)
    ext = Path(ruta).suffix.lower()
    nombre = os.path.basename(ruta)
    docs = []

    if ext == ".pdf":
        from langchain_community.document_loaders import PyPDFLoader
        for pagina in PyPDFLoader(ruta).load():
            texto = re.sub(r"\s+", " ", pagina.page_content)
            fuente = f"{nombre} p.{pagina.metadata.get('page', 0) + 1}"
            docs.append(Document(page_content=texto, metadata={"fuente": fuente}))

    elif ext in (".txt", ".md"):
        texto = re.sub(r"[ \t]+", " ", Path(ruta).read_text(encoding="utf-8"))
        docs.append(Document(page_content=texto, metadata={"fuente": nombre}))

    elif ext == ".csv":
        import csv
        with open(ruta, encoding="utf-8") as f:
            for n, fila in enumerate(csv.reader(f), start=1):
                linea = " | ".join(fila)
                if linea.strip():
                    docs.append(Document(page_content=linea, metadata={"fuente": f"{nombre} fila {n}"}))

    elif ext == ".docx":
        # Requiere docx2txt: pip install docx2txt
        from langchain_community.document_loaders import Docx2txtLoader
        for d in Docx2txtLoader(ruta).load():
            texto = re.sub(r"\s+", " ", d.page_content)
            docs.append(Document(page_content=texto, metadata={"fuente": nombre}))

    else:
        raise ValueError(f"Formato no soportado: {ext}. Usa pdf, txt, md, csv o docx.")

    return docs


def fragmentar(paginas, size, overlap):
    """Convierte paginas/secciones en fragmentos chunked, conservando la fuente."""
    frags = []
    for p in paginas:
        for t in dividir_texto(p.page_content, size, overlap):
            frags.append(Document(page_content=t, metadata={"fuente": p.metadata["fuente"]}))
    return frags


def nombre_indice(base):
    return f"{base}:idx"


def vector_a_bytes(vector):
    return np.asarray(vector, dtype=np.float32).tobytes()


def crear_indice_knn(base):
    """Crea un indice RediSearch HNSW/KNN sobre los fragmentos de la base."""
    indice = nombre_indice(base)
    try:
        R.ft(indice).dropindex(delete_documents=False)
        print(f"Indice anterior eliminado: {indice}")
    except Exception:
        pass

    schema = (
        TextField("texto"),
        TextField("fuente"),
        VectorField(
            CAMPO_VECTOR,
            ALGORITMO_VECTORIAL,
            {
                "TYPE": "FLOAT32",
                "DIM": DIM_EMBEDDINGS,
                "DISTANCE_METRIC": DISTANCIA,
            },
        ),
    )
    R.ft(indice).create_index(
        schema,
        definition=IndexDefinition(
            prefix=[f"{base}:fragmento:"],
            index_type=IndexType.HASH,
        ),
    )
    print(f"Indice KNN creado: {indice} | campo={CAMPO_VECTOR} dim={DIM_EMBEDDINGS} metrica={DISTANCIA}")


def guardar_fragmentos(base, fragmentos):
    """Carga fragmentos en Redis y crea un indice vectorial KNN.

    La firma incluye la version del formato para forzar reingesta si antes los
    vectores estaban guardados como JSON en vez de bytes FLOAT32.

    Importante: primero calcula embeddings y solo despues reemplaza Redis. Asi,
    si Gemini falla por creditos/cuota, no se borra una base existente.
    """
    firma_base = "knn-float32-v1|" + "|".join(f.page_content for f in fragmentos)
    firma = hashlib.md5(firma_base.encode()).hexdigest()
    if R.get(f"{base}:firma") == firma:
        print(f"[{base}] Sin cambios: no se reindexa ({R.scard(base + ':fragmentos')} fragmentos).")
        try:
            info = R.ft(nombre_indice(base)).info()
            print("Indice existente:", nombre_indice(base), "| docs:", info.get("num_docs"))
        except Exception as exc:
            print("Aviso: los fragmentos existen, pero falta el indice KNN:", exc)
            crear_indice_knn(base)
        return

    print(f"[{base}] Calculando embeddings antes de reemplazar Redis...")
    try:
        vectores = emb.embed_documents([f.page_content for f in fragmentos])
    except Exception as exc:
        print("No se pudo crear embeddings. No se borro ni reemplazo nada en Redis.")
        print("Causa:", type(exc).__name__, exc)
        raise

    # Solo si los embeddings se calcularon bien, se reemplaza esta base.
    try:
        R.ft(nombre_indice(base)).dropindex(delete_documents=False)
    except Exception:
        pass
    for k in R.scan_iter(f"{base}:fragmento:*"):
        R.delete(k)
    R.delete(f"{base}:fragmentos")

    crear_indice_knn(base)

    for i, (f, v) in enumerate(zip(fragmentos, vectores)):
        R.hset(f"{base}:fragmento:{i}", mapping={
            "texto": f.page_content,
            "fuente": f.metadata["fuente"],
            CAMPO_VECTOR: vector_a_bytes(v),
        })
        R.sadd(f"{base}:fragmentos", i)
    R.set(f"{base}:firma", firma)
    print(f"[{base}] {len(vectores)} vectores FLOAT32 guardados e indexados.")
print("Funciones listas.")


## 4. Parametros - EDITA ESTO

- `ARCHIVOS`: lista de documentos a cargar en el mismo corpus RAG.
- `NOMBRE_BASE`: subcarpeta dentro de tu prefijo en Redis. Para el asistente final debe ser `ebook_cripto`.
- `CHUNK_SIZE` / `CHUNK_OVERLAP`: tamano y solapamiento de los fragmentos.

Esta configuracion deja el corpus en `Grupo_1:ebook_cripto` si `REDIS_PREFIJO=Grupo_1`.
El indice vectorial queda en `Grupo_1:ebook_cripto:idx`.


In [ ]:
ARCHIVOS = [
    "criptomonedas-desde-cero.md",
    "Criptoactivos_Basico.pdf",
]
NOMBRE_BASE   = "ebook_cripto"  # base esperada por asistente_cripto.ipynb
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 75

BASE = f"{PREFIJO}:{NOMBRE_BASE}"
print("Documentos a cargar:")
for archivo in ARCHIVOS:
    existe = Path(archivo).exists()
    print(" -", archivo, "| existe:", existe)
    if not existe:
        raise FileNotFoundError(f"No se encontro el documento: {archivo}")
print("En la base Redis:", BASE)
print("Indice RediSearch:", nombre_indice(BASE))
print("Campos del hash: texto, fuente,", CAMPO_VECTOR)
print("Embedding:", MODELO_EMBEDDINGS, "| dim:", DIM_EMBEDDINGS)
print("Vector index:", ALGORITMO_VECTORIAL, "| metrica:", DISTANCIA)
print(f"Chunking: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")


## 5. Cargar el documento a Redis

In [ ]:
paginas = []
for archivo in ARCHIVOS:
    docs = cargar_documento(archivo)
    paginas.extend(docs)
    print(f"{archivo}: {len(docs)} secciones/paginas leidas")
print(f"Total secciones/paginas leidas: {len(paginas)}")

fragmentos = fragmentar(paginas, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Fragmentos tras el chunking: {len(fragmentos)}")

fuentes = sorted({f.metadata["fuente"] for f in fragmentos})
print("Fuentes detectadas en fragmentos:")
for fuente in fuentes[:20]:
    print(" -", fuente)
if len(fuentes) > 20:
    print(f" ... y {len(fuentes) - 20} fuentes/paginas mas")

guardar_fragmentos(BASE, fragmentos)
print("\nVerificacion en Redis ->", R.scard(f"{BASE}:fragmentos"), "fragmentos en", BASE)
print("Indice KNN ->", nombre_indice(BASE))
try:
    info = R.ft(nombre_indice(BASE)).info()
    print("Docs indexados ->", info.get("num_docs"))
    print("Definicion -> HASH prefix", f"{BASE}:fragmento:")
except Exception as exc:
    print("No se pudo leer FT.INFO del indice:", exc)


## 6. Prueba rapida de recuperacion KNN

Busca los fragmentos mas parecidos a una pregunta para confirmar que la carga funciono. Esta prueba **usa RediSearch/KNN en Redis**, no calcula similitud con NumPy en memoria.

La salida muestra distancia, parecido, fuente y un recorte del fragmento recuperado.


In [ ]:
def buscar(base, pregunta, k=3):
    q = vector_a_bytes(emb.embed_query(pregunta))
    consulta = (
        Query(f"*=>[KNN {k} @{CAMPO_VECTOR} $vec AS distancia]")
        .sort_by("distancia")
        .return_fields("texto", "fuente", "distancia")
        .dialect(2)
    )
    res = R.ft(nombre_indice(base)).search(consulta, query_params={"vec": q})
    salida = []
    for doc in res.docs:
        distancia = float(doc.distancia)
        salida.append({
            "texto": doc.texto,
            "fuente": doc.fuente,
            "distancia": round(distancia, 4),
            "parecido": round(1 - distancia, 4),
        })
    return salida

PREGUNTA = "que es blockchain?"   # <-- cambiala por una tuya
print("Pregunta de prueba:", PREGUNTA)
for r in buscar(BASE, PREGUNTA, k=3):
    print(f"[parecido={r['parecido']} distancia={r['distancia']}] {r['fuente']}")
    print("   ", r["texto"][:220], "...\n")


## 7. (Opcional) Ver o borrar bases en Redis

Útil para listar lo que tienes cargado o limpiar una base.

In [ ]:
# Listar las bases (subcarpetas) de tu prefijo:
bases = sorted({k.rsplit(":fragmentos", 1)[0] for k in R.scan_iter(f"{PREFIJO}:*:fragmentos")})
print("Bases en Redis bajo", PREFIJO, ":")
for b in bases:
    indice = nombre_indice(b)
    try:
        info = R.ft(indice).info()
        idx = f"indice={indice} docs={info.get('num_docs')}"
    except Exception:
        idx = "sin indice KNN"
    print("  -", b, "->", R.scard(f"{b}:fragmentos"), "fragmentos |", idx)

# Para BORRAR una base, descomenta y pon su nombre completo:
# base_a_borrar = f"{PREFIJO}:ebook_cripto"
# try:
#     R.ft(nombre_indice(base_a_borrar)).dropindex(delete_documents=False)
# except Exception:
#     pass
# for k in R.scan_iter(f"{base_a_borrar}:*"):
#     R.delete(k)
# print("Borrada:", base_a_borrar)
